In [1]:
from ASKpipeline import build_verification_graph
import pandas as pd
from collections import Counter
import json
import os

DOWNSAMPLE=True

/Users/cameronkruger/OTHELLO-1/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
from typing import Dict, List
import pandas as pd

def run_verification_on_datasets(
    datasets: Dict[str, pd.DataFrame],
    question_col: str = "question",
    answer_col: str = "answer",
    build_graph_fn=None,
    show_progress: bool = True,
) -> Dict[str, List[str]]:
    """
    Runs the verification graph on each row of each dataframe.
    Returns: {dataset_name: [verdicts...]} in row order.
    """
    if build_graph_fn is None:
        from ASKpipeline import build_verification_graph
        build_graph_fn = build_verification_graph

    graph = build_graph_fn()
    results: Dict[str, List[str]] = {}

    for name, df in datasets.items():
        verdicts: List[str] = []
        iterator = df.itertuples(index=False)
        if show_progress:
            try:
                from tqdm import tqdm
                iterator = tqdm(list(iterator), desc=f"Verifying {name}")
            except Exception:
                pass

        for row in iterator:
            row_dict = row._asdict()
            state = {
                "question": row_dict.get(question_col, ""),
                "answer": row_dict.get(answer_col, ""),
            }
            try:
                out = graph.invoke(state)
                verdicts.append(out.get("verdict", "Not Found"))
            except Exception:
                verdicts.append("ERROR")

        results[name] = verdicts

    return results


In [3]:
names = ['mintaka', 'qald', 'hotpot']

variants = ['small','base','large']
norm_files = [f'flan-t5-{variant}' for variant in variants]
vanilla_files = [f'vanilla-flan-t5-{variant}' for variant in variants]

files = norm_files + vanilla_files

file_name = files[0]


dataframes = {name: pd.read_csv(f'./LLM_answers/{file_name}/LLM_Answers_{name}.csv') for name in names}

if file_name.startswith('vanilla'):
    dataframes = {name: pd.read_csv(f'./LLM_answers/{file_name}/Vanilla_LLM_Answers_{name}.csv') for name in names}




In [8]:
print(len(dataframes.values()))

3


In [4]:
from ASKpipeline import build_verification_graph

state = {"queries": 'ASK WHERE {{ wd:Q5351150 wdt:P17 wd:Q49 . }UNION{ wd:Q49 wdt:P17 wd:Q5351150 . }}'}
graph = build_verification_graph(state)

row = dataframes["mintaka"].iloc[0]
state = {"question": row["SAE Question"], "answer": row["Answer"]}
out = graph.invoke(state)



parsed_entities: {0: ['El Diente Peak', 'North America']}
parsed_relations: {0: ['height', 'rank', 'location', 'part of', 'country']}
entity_qids: {0: ['Q5351150', 'Q49']}
relation_pids: {0: ['P2044', 'P1545', 'P625', 'P361', 'P17']}
queries: ['ASK WHERE {{ wd:Q5351150 wdt:P2044 wd:Q49 . }UNION{ wd:Q49 wdt:P2044 wd:Q5351150 . }}']
results: [False]
results_sum: 0


In [5]:
type(out['results'][0])

bool

In [ ]:
all_results = {}

for file_name in files:
    if file_name.startswith("vanilla"):
        dataframes = {
            name: pd.read_csv(f"./LLM_answers/{file_name}/Vanilla_LLM_Answers_{name}.csv")
            for name in names
        }
    else:
        dataframes = {
            name: pd.read_csv(f"./LLM_answers/{file_name}/LLM_Answers_{name}.csv")
            for name in names
        }

    file_results = {}

    for name, data in dataframes.items():
        print(f"Starting verification for {name}")
        if DOWNSAMPLE:
            print(f"Truncating {name} to 10 items")
            data = data[:20]

        row_results = []
        trues, falses = 0, 0

        for _, row in data.iterrows():
            state = {"question": row["SAE Question"], "answer": row["Answer"]}
            print(f"state:\t{state}")
            out = graph.invoke(state)
            results = out.get("results") or []

            row_bool = any(results) #Returns true if at least one entry is true

            row_results.append(row_bool)
            trues += int(row_bool)
            falses += int(not row_bool)

        total = trues + falses
        if total:
            true_pct = (trues / total) * 100
            false_pct = (falses / total) * 100
        else:
            true_pct = None
            false_pct = None

        print(
            f"{name} totals — Trues: {trues}, Falses: {falses}, "
            f"True percent: {true_pct}, False percent: {false_pct}"
        )

        fin_res = pd.DataFrame({"Results": row_results})

        file_results[name] = {
            "rows": fin_res.to_dict(orient="records"),
            "totals": {"Trues": trues, "Falses": falses},
            "pct_totals": {"percent_true": true_pct, "percent_false": false_pct},
        }

    out_dir = f"./final_results/{file_name}"
    os.makedirs(out_dir, exist_ok=True)

    json_path = f"{out_dir}/{file_name}_results.json"
    with open(json_path, "w") as f:
        json.dump(file_results, f, indent=2)

    all_results[file_name] = file_results

all_results_path = "./final_results/all_results.json"
with open(all_results_path, "w") as f:
    json.dump(all_results, f, indent=2)


Starting verification for mintaka
Truncating mintaka to 10 items
state:	{'question': 'What is the seventh tallest mountain in North America?', 'answer': 'El Diente Peak'}
parsed_entities: {0: ['El Diente Peak', 'North America']}
parsed_relations: {0: ['tallest mountain', 'location', 'part of mountain range', 'elevation', 'rank in height']}
entity_qids: {0: ['Q5351150', 'Q49']}
relation_pids: {0: ['P625', 'P2044']}
queries: ['ASK WHERE {{ wd:Q5351150 wdt:P625 wd:Q49 . }UNION{ wd:Q49 wdt:P625 wd:Q5351150 . }}']
results: [False]
results_sum: 0
state:	{'question': 'Which actor was the star of Titanic and was born in Los Angeles, California?', 'answer': "I don't know"}
parsed_entities: {0: [], 1: ['Leonardo DiCaprio']}
parsed_relations: {0: [], 1: ['P19', 'P19', 'P31']}
entity_qids: {0: [], 1: ['Q38111']}
relation_pids: {0: [], 1: ['P19', 'P19', 'P31']}
queries: []
results: []
results_sum: 0
state:	{'question': 'Which actor starred in Vanilla Sky and was married to Katie Holmes?', 'answer':

In [52]:
for k1 in all_results.keys():
    for k2 in all_results[k1].keys():
        # for k3 in all_results[k1][k2].keys():
        print(all_results[k1][k2])
    

[{'Results': ['False'], 'Majority': 'None'}, {'Results': ['None', 'False'], 'Majority': 'None'}, {'Results': ['None', 'False', 'None'], 'Majority': 'None'}, {'Results': ['None'], 'Majority': 'None'}, {'Results': ['None', 'None'], 'Majority': 'None'}]
[{'Results': ['True'], 'Majority': 'None'}, {'Results': ['None', 'None'], 'Majority': 'None'}, {'Results': ['None'], 'Majority': 'None'}, {'Results': [], 'Majority': 'None'}, {'Results': ['False'], 'Majority': 'None'}]
[{'Results': ['None', 'False'], 'Majority': 'None'}, {'Results': ['False', 'None'], 'Majority': 'None'}, {'Results': ['False', 'None'], 'Majority': 'None'}, {'Results': ['None'], 'Majority': 'None'}, {'Results': ['None', 'None'], 'Majority': 'None'}]
[{'Results': ['None'], 'Majority': 'None'}, {'Results': ['None', 'None'], 'Majority': 'None'}, {'Results': ['None', 'None'], 'Majority': 'None'}, {'Results': ['False'], 'Majority': 'None'}, {'Results': ['None', 'False'], 'Majority': 'None'}]
[{'Results': ['True'], 'Majority': 'F

In [54]:
# import json

with open(all_results_path, "w") as f:
    json.dump(all_results, f, indent=2)
